# 03 — Departure Classification Baseline (V9.0)

**LGB + XGB + CatBoost baselines → Three-Model Comparison → Threshold Preview → Save Context**

In [1]:
# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# ML imports
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# === Project paths (RCF: only change BASE_DIR) ===
BASE_DIR = Path('../../..')             # RCF: Path('/path/to/project')
if not (BASE_DIR / 'src').exists():
    BASE_DIR = Path('../../..')
sys.path.insert(0, str(BASE_DIR / 'src'))

from features.lag_features import add_lag_features, add_congestion_features, compute_v4_lag_features
from features.aircraft_features import compute_prev_aircraft_delay
from models.threshold_optimizer import (
    find_optimal_threshold, evaluate_at_threshold,
    plot_threshold_analysis, get_business_metrics
)

# Paths
PROJECT_ROOT = BASE_DIR
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
FIGURES_DIR = Path('../../../outputs/figures/departure')
MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports complete (LGB + XGB + CatBoost + threshold optimizer)')

Imports complete (LGB + XGB + CatBoost + threshold optimizer)


## 1. Load Data from NB02

In [2]:
# === Load featured data from 02 ===
train = pd.read_parquet(DATA_PROCESSED / 'dep_train_featured.parquet')
test = pd.read_parquet(DATA_PROCESSED / 'dep_test_featured.parquet')
print(f'Loaded: train={len(train):,}, test={len(test):,}')
print(f'Delay rate: train={train["Is_Delayed"].mean()*100:.1f}%, test={test["Is_Delayed"].mean()*100:.1f}%')

Loaded: train=104,460, test=44,117
Delay rate: train=21.5%, test=17.8%


## 2. Feature Columns (22 features)

In [3]:
# Feature columns for V9.0 departure model (23 features)
# Must match 02_feature_engineering.ipynb feature_columns

feature_columns = [
    # Core Departure Lag (5)
    'delay_rate_1h',               # past 1h departure delay rate
    'delay_rolling_3h',            # past 3h departure rolling mean
    'severe_delay_count_prev',     # past 3h severe departure count
    'terminal_delay_1h',           # terminal-specific departure delay
    'dep_runway_config_change',    # departure runway changed

    # Cross-modal Lag (1)
    'lga_arr_delay_1h',            # arrival congestion -> departure delay

    # Aircraft Continuity (2)
    'prev_inbound_delay',          # previous arrival delay of same aircraft
    'turnaround_hours',            # gap between inbound arrival and outbound departure

    # Target Encoding (4)
    'dep_gate_delay_rate',         # gate-level delay pattern
    'dep_airline_delay_rate',      # airline delay pattern
    'dep_runway_delay_rate',       # runway delay pattern
    'dep_faa_delay_reason',        # FAA severity

    # Time (2)
    'Hour',
    'Month',

    # FAA (1)
    'faa_delay_severity',

    # Weather Impact Scores (2)
    'dest_wx_impact',              # destination weather impact (0-10)
    'lga_wx_impact',               # LGA weather impact

    # Destination Weather (2)
    'dest_dewpoint',
    'dest_pressure_change_3h',

    # Destination Historical (1)
    'dest_historical_delay',

    # FAA Event Depth (2)
    'faa_event_duration_hours',
    'faa_active_event_count',

    # Network (1)
    'route_risk_score',            # destination route risk (V9.0)
]
print(f'Departure feature_columns: {len(feature_columns)} features (V9.0)')

# Prepare X/y splits
available = [c for c in feature_columns if c in train.columns]
missing = [c for c in feature_columns if c not in train.columns]
if missing:
    print(f'WARNING: Missing features: {missing}')
feature_columns = available

X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()
y_train = train['Is_Delayed'].copy()
y_test = test['Is_Delayed'].copy()

train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Delay rate: train={y_train.mean()*100:.1f}%, test={y_test.mean()*100:.1f}%')

Departure feature_columns: 23 features (V9.0)
X_train: (104460, 23), X_test: (44117, 23)
Delay rate: train=21.5%, test=17.8%


## 3. LGB Baseline

In [4]:
# LGB baseline with auto scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
base_spw = neg_count / pos_count
print(f'Class imbalance ratio (scale_pos_weight): {base_spw:.2f}')

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    scale_pos_weight=base_spw,
    random_state=SEED,
    verbose=-1,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_proba)
print(f'LGB baseline AUC: {lgb_auc:.4f}')

Class imbalance ratio (scale_pos_weight): 3.66
LGB baseline AUC: 0.8808


## 4. XGB Baseline

In [5]:
# XGB baseline with same scale_pos_weight
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    scale_pos_weight=base_spw,
    random_state=SEED,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='logloss',
)
xgb_model.fit(X_train, y_train)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_proba)
print(f'XGB baseline AUC: {xgb_auc:.4f}')

XGB baseline AUC: 0.8801


## 5. CatBoost Baseline (Balanced)

In [6]:
# CatBoost baseline with auto_class_weights='Balanced'
cb_model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    random_seed=SEED,
    verbose=0,
)
cb_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=0,
)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
cb_auc = roc_auc_score(y_test, cb_proba)
print(f'CatBoost baseline AUC: {cb_auc:.4f}')
print(f'Best iteration: {cb_model.best_iteration_}')

CatBoost baseline AUC: 0.8868
Best iteration: 299


## 6. Three-Model Comparison (LGB vs XGB vs CatBoost)

In [7]:
# --- AUC comparison ---
models = {
    'LightGBM baseline': (lgb_proba, lgb_auc),
    'XGBoost baseline': (xgb_proba, xgb_auc),
    'CatBoost baseline': (cb_proba, cb_auc),
}

print('=== Baseline AUC Comparison ===')
print(f'{"Model":<25s} {"AUC":>8s}')
print('-' * 35)
for name, (proba, auc) in models.items():
    print(f'{name:<25s} {auc:>8.4f}')

# Best baseline
best_name = max(models, key=lambda k: models[k][1])
print(f'\nBest baseline: {best_name} (AUC={models[best_name][1]:.4f})')

=== Baseline AUC Comparison ===
Model                          AUC
-----------------------------------
LightGBM baseline           0.8808
XGBoost baseline            0.8801
CatBoost baseline           0.8868

Best baseline: CatBoost baseline (AUC=0.8868)


In [8]:
# --- Detailed comparison at multiple thresholds ---
print('\nComparison at threshold = 0.50:')
print(f'{"Model":<25s} {"AUC":>7s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 55)
for name, (proba, auc) in models.items():
    preds = (proba >= 0.50).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    print(f'{name:<25s} {auc:>7.4f} {p:>7.1%} {r:>7.1%} {f:>7.1%}')

# Threshold at R>=65%
print(f'\nAt target recall >= 65%:')
print(f'{"Model":<25s} {"Thresh":>7s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 55)
for name, (proba, auc) in models.items():
    t, m = find_optimal_threshold(y_test.values, proba, target_recall=0.65, min_precision=0.30)
    if t is not None:
        print(f'{name:<25s} {t:>7.2f} {m["precision"]:>7.1%} {m["recall"]:>7.1%} {m["f1"]:>7.1%}')
    else:
        print(f'{name:<25s} {"N/A":>7s}')


Comparison at threshold = 0.50:
Model                         AUC    Prec  Recall      F1
-------------------------------------------------------
LightGBM baseline          0.8808   71.3%   69.4%   70.3%
XGBoost baseline           0.8801   74.5%   68.5%   71.4%
CatBoost baseline          0.8868   70.0%   70.5%   70.2%

At target recall >= 65%:
Model                      Thresh    Prec  Recall      F1
-------------------------------------------------------
LightGBM baseline            0.59   77.9%   67.1%   72.1%
XGBoost baseline             0.59   80.6%   65.9%   72.5%
CatBoost baseline            0.59   76.5%   67.6%   71.7%


## 7. Threshold Optimization (Preview)

In [9]:
# Find optimal thresholds for each baseline model
for name, (proba, auc) in models.items():
    print(f'\n=== {name} ===')
    for mode_name, kwargs in [
        ('Balanced',       {'target_recall': 0.65, 'min_precision': 0.30}),
        ('High Precision', {'target_recall': 0.50, 'min_precision': 0.45}),
        ('High Recall',    {'target_recall': 0.80, 'min_precision': 0.20}),
    ]:
        t, m = find_optimal_threshold(y_test.values, proba, **kwargs)
        if t is not None:
            print(f'  {mode_name:15s}: t={t:.2f}, P={m["precision"]:.1%}, R={m["recall"]:.1%}, F1={m["f1"]:.1%}')
        else:
            print(f'  {mode_name:15s}: NOT FOUND')


=== LightGBM baseline ===
  Balanced       : t=0.59, P=77.9%, R=67.1%, F1=72.1%
  High Precision : t=0.59, P=77.9%, R=67.1%, F1=72.1%
  High Recall    : t=0.25, P=43.4%, R=80.4%, F1=56.4%

=== XGBoost baseline ===
  Balanced       : t=0.59, P=80.6%, R=65.9%, F1=72.5%
  High Precision : t=0.59, P=80.6%, R=65.9%, F1=72.5%
  High Recall    : t=0.24, P=43.7%, R=80.5%, F1=56.6%

=== CatBoost baseline ===
  Balanced       : t=0.59, P=76.5%, R=67.6%, F1=71.7%
  High Precision : t=0.59, P=76.5%, R=67.6%, F1=71.7%
  High Recall    : t=0.28, P=47.2%, R=80.4%, F1=59.5%


## 8. Historical Comparison

V8.0 departure reference: CatBoost Optuna AUC=0.8917, P=79.7%, R=66.1%, F1=72.3%

In [10]:
# Historical comparison table
comparison = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Precision', 'Recall', 'F1', 'Features'],
    'V7.0 (21 feat)': [0.8934, '79.7%', '66.1%', '72.3%', 21],
    'V8.0 Ref': [0.8917, '79.7%', '66.1%', '72.3%', 23],
})

# Add current baselines
for name, (proba, auc) in models.items():
    t, m = find_optimal_threshold(y_test.values, proba, target_recall=0.65, min_precision=0.30)
    short_name = name.split()[0] + ' BL'
    if t is not None:
        comparison[f'V9.0 {short_name}'] = [
            round(auc, 4),
            f'{m["precision"]:.1%}',
            f'{m["recall"]:.1%}',
            f'{m["f1"]:.1%}',
            len(feature_columns),
        ]

print(comparison.to_string(index=False))

   Metric V7.0 (21 feat) V8.0 Ref V9.0 LightGBM BL V9.0 XGBoost BL V9.0 CatBoost BL
  AUC-ROC         0.8934   0.8917           0.8808          0.8801           0.8868
Precision          79.7%    79.7%            77.9%           80.6%            76.5%
   Recall          66.1%    66.1%            67.1%           65.9%            67.6%
       F1          72.3%    72.3%            72.1%           72.5%            71.7%
 Features             21       23               23              23               23


## 9. Save Context

In [11]:
# === Save prepared data for downstream notebooks (04, 05) ===
import pickle

dep_baseline_context = {
    'feature_columns': feature_columns,
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'train': train, 'test': test,
    'train_medians': train_medians,
    # Baseline models & probas (for comparison in 05)
    'lgb_model': lgb_model, 'lgb_proba': lgb_proba, 'lgb_auc': lgb_auc,
    'xgb_model': xgb_model, 'xgb_proba': xgb_proba, 'xgb_auc': xgb_auc,
    'cb_model': cb_model, 'cb_proba': cb_proba, 'cb_auc': cb_auc,
}
pickle.dump(dep_baseline_context, open(DATA_PROCESSED / 'dep_baseline_context.pkl', 'wb'))
print(f'Saved dep_baseline_context.pkl: {X_train.shape[1]} features, {len(train):,} train, {len(test):,} test')

Saved dep_baseline_context.pkl: 23 features, 104,460 train, 44,117 test


In [12]:
# Final summary
print('=' * 60)
print('DEPARTURE BASELINE TRAINING COMPLETE (V9.0)')
print('=' * 60)
print(f'\nFeatures: {len(feature_columns)}')
print(f'Train: {len(train):,} ({y_train.mean()*100:.1f}% delayed)')
print(f'Test:  {len(test):,} ({y_test.mean()*100:.1f}% delayed)')
print(f'\nBaseline AUC:')
print(f'  LGB:      {lgb_auc:.4f}')
print(f'  XGB:      {xgb_auc:.4f}')
print(f'  CatBoost: {cb_auc:.4f}')
print(f'\nContext saved → run 05_deployment.ipynb for Optuna tuning')

DEPARTURE BASELINE TRAINING COMPLETE (V9.0)

Features: 23
Train: 104,460 (21.5% delayed)
Test:  44,117 (17.8% delayed)

Baseline AUC:
  LGB:      0.8808
  XGB:      0.8801
  CatBoost: 0.8868

Context saved → run 05_deployment.ipynb for Optuna tuning
